In [2]:
# Project 2 Animal Front End Database
# Cranmer, Thomas
# 13Aug26

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Python import
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Database credentials and database interaction
username = "aacuser"
password = "Kazters2!"
shelter = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.query({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Animal Database'))),
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style= {
                "display": "block",
                "marginLeft": "auto",
                "marginRight": "auto",
                "width": "15%"
            }),
    html.Center(html.H1("Cranmer, Thomas and the CRUD Implementation")),
    html.Hr(),
    html.Div(
        
        dcc.Dropdown(["All", "Water Rescue", "Mountain Rescue", "Disaster Rescue"], value='All', id='Dog-filtering')
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         row_selectable = "single",
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         selected_columns = [],
                         selected_rows = [0],
                         row_deletable = False,
                         page_size = 20,
                         page_current = 0,

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('Dog-filtering', 'value')])
def update_dashboard(filter_type):

    # If dropdown is Water Rescue
    if filter_type == 'Water Rescue':
        dff = df[(df['breed'].isin(['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland'])) &
                (df['sex_upon_outcome'] == 'Intact Female') &
                (df['age_upon_outcome_in_weeks'] >= 26) &
                (df['age_upon_outcome_in_weeks'] <= 156)] 
        
        update_graphs(dff) # calls update graph to update pie chart
    
    # If dropdown is Mountain Rescue
    elif filter_type == 'Mountain Rescue':
        dff = df[(df['breed'].isin(['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler'])) &
                (df['sex_upon_outcome'] == 'Intact Male') &
                (df['age_upon_outcome_in_weeks'] >= 26) &
                (df['age_upon_outcome_in_weeks'] <= 156)] 
        
        update_graphs(dff) # calls update graph to update pie chart
    
    # If dropdown is Disaster Rescue
    elif filter_type == 'Disaster Rescue':
        dff = df[(df['breed'].isin(['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler'])) &
                (df['sex_upon_outcome'] == 'Intact Male') &
                (df['age_upon_outcome_in_weeks'] >= 20) &
                (df['age_upon_outcome_in_weeks'] <= 300)] 
        
        update_graphs(dff) # calls update graph to update pie chart
    elif filter_type == 'All':
        dff = df #Resets dataframe
        update_graphs(dff) # calls update graph to update pie chart
        
        
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in dff.columns]
    data=dff.to_dict('records')
       
      
    return data
        

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    
    if viewData is None or len(viewData) == 0:
        return "No data passed"
    
    dff = pd.DataFrame.from_dict(viewData)
    
    if dff.empty:
        return "No data to be displayed"
    
    return dcc.Graph(            
                figure = px.pie(dff, names='breed', title='Preferred Animals')
            )
    
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    # Majority function provided by milestone prompt
     # Because we only allow single row selection, the list can 
     # be converted to a row index here
    
    # Check for no viewData passed, sets default ouput
    if viewData is None or len(viewData) == 0:
        testCenter = [30.75, -97.48] # Austin, TX
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'},
                   center=testCenter, zoom=10, children=[
                       dl.Tilelayer(id="base-layer-id")
                   ])
        ]
    
    # Converts dictionary style data set to Data Frame
    dff = pd.DataFrame.from_dict(viewData)
    
    # Checks if the conversion above is empty, sets default output
    if dff.empty:
        testCenter = [30.75, -97.48] # Austin, TX
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'},
                   center=testCenter, zoom=10, children=[
                       dl.Tilelayer(id="base-layer-id")
                   ])
        ]
    
    # Checks for any instance of the index not existing or being empty
    if index is None or len(index) == 0 or index[0] >= len(dff):
       row = 0
    else: 
       row = index[0]
    
    # Checks that there are at least 14 key/value pairs converted for proper display, otherwise throws exception
    if len(dff.columns) > 14:
        try:
            latitude = dff.iloc[row, 13]
            longitude = dff.iloc[row, 14]
            
            if longitude is None or latitude is None:
                testCenter = [30.75, -97.58]
            else:
                testCenter = [float(latitude), float(longitude)]
        except Exception as ex:
            testCenter = [30.75, -97.58]
    else:
        testCenter = [30.75, -97.58]
    
    # Ensures message is displayed in map point/popup info on map for any potential empty index points
    toolTipInfo = dff.iloc[row,4] if len(dff.columns) > 4 else "Breed not found"
    petName = dff.iloc[row,9] if len(dff.columns) > 9 else "Pet has no name"

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
           center=testCenter, zoom=10, children=[
               dl.TileLayer(id="base-layer-id"),
           # Marker with tool tip and popup
           # Column 13 and 14 define the grid-coordinates for 
           # the map
           # Column 4 defines the breed for the animal
           # Column 9 defines the name of the animal
               dl.Marker(position=testCenter,
                  children=[
                      dl.Tooltip(toolTipInfo),
                      dl.Popup([
                         html.H1("Pet's Name"),
                         html.P(petName)
                     ])
                  ])
           ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

{'_id': ObjectId('6a5a8422561852c04392209a'),
 'age_upon_outcome': '2 years',
 'age_upon_outcome_in_weeks': 110.111408730159,
 'animal_id': 'A716330',
 'animal_type': 'Dog',
 'breed': 'Chihuahua Shorthair Mix',
 'color': 'Brown/White',
 'date_of_birth': '2013-11-18',
 'datetime': '2015-12-28 18:43:00',
 'location_lat': 30.7595748121648,
 'location_long': -97.5523753807133,
 'monthyear': '2015-12-28T18:43:00',
 'name': 'Frank',
 'outcome_subtype': '',
 'outcome_type': 'Adoption',
 'rec_num': 3,
 'sex_upon_outcome': 'Neutered Male'}
{'_id': ObjectId('6a5a8422561852c04392209b'),
 'age_upon_outcome': '1 year',
 'age_upon_outcome_in_weeks': 52.9215277777778,
 'animal_id': 'A725717',
 'animal_type': 'Cat',
 'breed': 'Domestic Shorthair Mix',
 'color': 'Silver Tabby',
 'date_of_birth': '2015-05-02',
 'datetime': '2016-05-06 10:49:00',
 'location_lat': 30.6525984560228,
 'location_long': -97.7419963476444,
 'monthyear': '2016-05-06T10:49:00',
 'name': '',
 'outcome_subtype': 'SCRP',
 'outcome_

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



{'_id': ObjectId('6a5a8422561852c0439241e5'),
 'age_upon_outcome': '2 years',
 'age_upon_outcome_in_weeks': 105.537103174603,
 'animal_id': 'A666957',
 'animal_type': 'Dog',
 'breed': 'Doberman Pinsch',
 'color': 'Black/Tan',
 'date_of_birth': '2011-11-10',
 'datetime': '2013-11-17 18:14:00',
 'location_lat': 30.6317556266147,
 'location_long': -97.2982430445575,
 'monthyear': '2013-11-17T18:14:00',
 'name': 'Leroy',
 'outcome_subtype': '',
 'outcome_type': 'Adoption',
 'rec_num': 8530,
 'sex_upon_outcome': 'Neutered Male'}
{'_id': ObjectId('6a5a8422561852c0439241e6'),
 'age_upon_outcome': '2 years',
 'age_upon_outcome_in_weeks': 130.37619047619,
 'animal_id': 'A730981',
 'animal_type': 'Dog',
 'breed': 'Chihuahua Shorthair Mix',
 'color': 'Red',
 'date_of_birth': '2014-01-14',
 'datetime': '2016-07-14 15:12:00',
 'location_lat': 30.618557361027,
 'location_long': -97.4143792049517,
 'monthyear': '2016-07-14T15:12:00',
 'name': 'Red',
 'outcome_subtype': '',
 'outcome_type': 'Return to